# Experiment: PiP-noPlan (planning-informed prediction without a plan)

Song et al., ECCV 2020, run without its planning input (NGSIM has no ego plan): every vehicle's LSTM encoding is placed in a dense target-centred social tensor, a fully convolutional network fuses it at two scales, and a manoeuvre-based decoder predicts. GNP Table 1 (PiP): 0.55 / 1.18 / 1.94 / 2.88 / 4.04 m; PiP-noPlan is reported at 4.27 m at 5 s.

Part of the model comparison in `experiments/MODEL_ALTERNATIVES.md`. Run All; set `DEBUG = False` in section 2 for the real run.

## 0. Decisions & assumptions

Every choice that could change the result is listed here; the cell notes below repeat the ones that apply to each cell.

**Shared by every notebook (S)**

| # | Decision / assumption | Why | Departure from paper / alternative |
|---|---|---|---|
| S1 | Same data for every model: NGSIM us-101 + i-80 (DEBUG: first 250k rows of us-101), scene-level 70/10/20 split, seed 42, train-only normalization. | Differences in the results must come from the model, not the data. | Papers use their own splits, so published numbers are not like-for-like with ours. |
| S2 | Same training budget for every model: config.py defaults (12 epochs, Adam lr 5e-4 decayed x0.6 per epoch, batch 128, weight decay 1e-5, grad clip 1.0). | A fair comparison; per-model tuning would favour whichever model got tuned most. | Each paper's own schedule (e.g. CS-LSTM trains longer with lr 1e-3). Tune later with --mode tune if a model looks under-trained. |
| S3 | Loss = MSE + bivariate-Gaussian NLL + neighbour-goal MSE (`basic_loss`); the risk-scaled loss is used only where this notebook's M table says so (STRAP-R-based variants). Models without a goal head output zero goals, so their goal term is a constant. A model may add its own `aux_loss` (only the diffusion model does). | Keeps one training objective; the risk-scaled loss is STRAP-R's own contribution. | Loss values are therefore not comparable across models: compare RMSE only. |
| S4 | Best checkpoint = lowest validation loss (`fit` default); test RMSE is reported at 1-5 s for that checkpoint. | Same selection rule for every model. | Selecting on val RMSE would be closer to the reported metric. |
| S5 | KNOWN LIMITATION (not fixed): positions are relative to the target at each timestep, so the target's own history is always (0, 0) and lateral speed is always 0 (NGSIM gives longitudinal speed only). The target is seen through speed, acceleration and lane features; lateral context comes from neighbours. | Kept as is (user decision) so every model gets identical inputs. | Papers feed the target's own x/y track; lane-change prediction is handicapped for all models here. |
| S6 | Neighbours = up to 15 risk-selected vehicles (S/O-field > RISK_THRESHOLD), not every nearby vehicle. | Same scene construction as STRAP. | Grid/graph papers use all vehicles in range; ours see only the risk-selected ones. |
| S7 | Output = one bivariate Gaussian per future step (mu_x, mu_y, sigma_x, sigma_y, rho); RMSE uses the mean. Multimodal models are moment-matched to one Gaussian. | The `fit` / `evaluate_trajectory` contract and how STRAP's Table I is scored. | minADE/minFDE over modes would favour multimodal models. |

**Specific to this model (M)**

| # | Decision / assumption | Why | Departure from paper / alternative |
|---|---|---|---|
| M1 | No planning module: NGSIM gives no ego plan. | The paper's own PiP-noPlan ablation. | Full PiP needs the ego vehicle's future plan. |
| M2 | Social tensor 25 x 5: 25 longitudinal cells of 4.57 m (+-57 m) x 5 lanes (relative lane -2..+2). | PiP uses a wider target-centred tensor than CS-LSTM. | ASSUMPTION: cell size kept from CS-LSTM; PiP's exact tensor size not checked. |
| M3 | FCN: two conv branches (3x3 and 5x3 kernels, 64 channels) -> concat -> conv 3x3 -> global max + mean pooling; plus the target encoding. | Multi-scale fully convolutional fusion. | - |
| M4 | 6 manoeuvre modes, label-supervised. | PiP's manoeuvre-based decoder. | - |
| M5 | Manoeuvre classes are supervised through `aux_loss` with labels from the true future: lane change if the 5 s lateral offset exceeds half a lane (1.85 m), braking if the mean future speed is below 0.8 x the current speed; cross-entropy weight 1. | The paper trains its manoeuvre head on labels; the `aux_loss` hook is the only place the true future reaches the model. | ASSUMPTION: papers label lane changes from lane IDs within +-4 s; ours uses the lateral offset at 5 s. |

## 1. Setup

**Decisions in this cell:** seed fixed to `SEED` (42) in every notebook so initialisation is not a source of differences (S1).

In [ ]:
import sys
from pathlib import Path

# Repo root = first folder upwards that contains src/ (works from experiments/ or the root).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.config import (
    BATCH_SIZE, BETA_LOSS, DEBUG_MAX_ROWS, DEVICE, DROPOUT, DT, LEARNING_RATE, LR_DECAY, N_V,
    NGSIM_LOCATIONS, NUM_EPOCHS, NORMALIZATION_FILE, SEED, STATE_DIM, TF_STEPS, TH_STEPS, WEIGHT_DECAY,
)
from src.data.loader import NGSIMDataLoader, create_dataloader
from src.data.normalization import load_stats, normalize_samples
from src.evaluation.evaluate import evaluate_trajectory
from src.models.strap import STRAP
from src.pipeline import collision_scores, metric_rows, test_groups
from src.train import fit, prepare_data
from experiments.benchmark import compare, save_result

torch.manual_seed(SEED)
print("device:", DEVICE)

## 2. Experiment config

**Decisions in this cell:** training budget is config.py's (S2); `use_risk_loss` as in the M table; collisions scored at 2 m like `--mode collisions`.

In [ ]:
EXP_NAME = "pip_noplan"          # unique per notebook -> experiments/results/<EXP_NAME>.json
DESCRIPTION = "PiP-noPlan: target-centred 25x5 social tensor, multi-scale FCN, manoeuvre-supervised"

DEBUG = True                  # True: DEBUG_MAX_ROWS of us-101 for a quick check; False: full data
LOCATIONS = ["us-101"] if DEBUG else NGSIM_LOCATIONS

HPARAMS = dict(               # keyword args of src.train.fit
    num_epochs=NUM_EPOCHS,
    use_risk_loss=False,       # False = basic loss (STRAP-B style)
    beta=BETA_LOSS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    lr_decay=LR_DECAY,
    weight_decay=WEIGHT_DECAY,
)

RUN_COLLISIONS = True         # per-pair collision scoring on the test split (slow on full data)
COLLISION_THRESHOLD = 2.0     # metres, GT collision label distance

CKPT_DIR = ROOT / "experiments" / "checkpoints" / EXP_NAME

## 3. Data
Train-only normalization stats and the intention codebook are written to `CKPT_DIR`.

**Decisions in this cell:** same loader, split and train-only normalization as every other notebook (S1, S6); `norm_stats` is loaded here so the model (next section) can recover metres and lane numbers.

In [ ]:
loader = NGSIMDataLoader(locations=LOCATIONS, max_rows=DEBUG_MAX_ROWS if DEBUG else None)
train_s, val_s, centers = prepare_data(loader, save_dir=str(CKPT_DIR))
_, _, test_raw = loader.get_splits()   # cached; raw (unnormalized) test split
# Train-only mean/std; models that need metres or lane numbers undo the normalization with it.
norm_stats = load_stats(str(CKPT_DIR / NORMALIZATION_FILE))
print(f"train/val/test = {len(train_s)}/{len(val_s)}/{len(test_raw)}")

## 4. Model

Any `torch.nn.Module` works if `forward(states, mask, dims)` returns a 3-tuple:

| output | shape | meaning |
|---|---|---|
| `traj_dist` | `(B, T_f=50, 5)` | `[mu_x, mu_y, sigma_x, sigma_y, rho]` of the target, metres relative to its last observed position (x longitudinal, y lateral); `sigma > 0`, `rho` in (-1, 1) |
| `goals` | `(B, N-1, 4)` | neighbour goals; a model without a goal head returns `states.new_zeros(B, N - 1, 4)` |
| third | anything | ignored by training/eval (STRAP returns its risk field) |

Inputs: `states (B, T_h=30, N, 11)` normalized (vehicle 0 = target, rest neighbours, zero-padded),
`mask (B, N)` True = real vehicle, `dims (B, N, 2)` raw length/width in metres.
Feature index: 0 rel-longitudinal m, 1 rel-lateral m, 2 speed/30, 3 lateral speed (always 0), 4 accel/10,
5 type/2, 6 relative lane (target - vehicle, left = +), 7 length/10, 8 width/3, 9 S-field risk, 10 O-field risk.
Raw value = `states * std + mean` with `norm_stats`. STRAP's k-means codebook is only loaded if the model has a `risk_decoder`.

**Decisions in this cell:** no plan (M1), 25 x 5 tensor (M2), multi-scale FCN (M3), manoeuvre decoder (M4, M5). Built after the data cell because it needs `norm_stats`.

In [ ]:
def moment_match(raw, weights):
    """(B, M, T, 5) raw mode outputs + (B, M) mode weights -> (B, T, 5) one Gaussian (S7)."""
    mu = raw[..., :2]
    sig = F.softplus(raw[..., 2:4]) + 1e-3
    rho = torch.tanh(raw[..., 4])
    w = weights[:, :, None]                                         # (B, M, 1)
    mean = (w[..., None] * mu).sum(1)                               # (B, T, 2)
    d = mu - mean[:, None]
    var = (w[..., None] * (sig ** 2 + d ** 2)).sum(1)
    cov = (w * (rho * sig[..., 0] * sig[..., 1] + d[..., 0] * d[..., 1])).sum(1)
    sx, sy = var[..., 0].sqrt(), var[..., 1].sqrt()
    return torch.stack([mean[..., 0], mean[..., 1], sx, sy, (cov / (sx * sy)).clamp(-0.999, 0.999)], dim=-1)

class VehicleEncoder(nn.Module):
    """Shared per-vehicle encoder: FC + leaky ReLU + LSTM over the 30 history steps."""
    def __init__(self, input_dim=STATE_DIM, emb=32, d=64):
        super().__init__()
        self.embed = nn.Linear(input_dim, emb)
        self.lstm = nn.LSTM(emb, d, batch_first=True)

    def forward(self, x):                                           # x (B, T, N, F)
        B, T, N, D_in = x.shape
        seq, (h, _) = self.lstm(F.leaky_relu(self.embed(x.transpose(1, 2).reshape(B * N, T, D_in)), 0.1))
        return seq.reshape(B, N, T, -1), h[-1].reshape(B, N, -1)   # per-step and last-step encodings

def social_grid(raw_last, mask, h, rows=13, cols=3, cell=4.572):
    """Scatter neighbour vectors h (B, N, C) into a (B, C, rows, cols) grid by longitudinal offset
    (feature 0) and relative lane (feature 6, left = +) at the last step; the target is left out."""
    B, N, C = h.shape
    row = torch.floor(raw_last[..., 0] / cell + rows / 2).long()
    col = (cols // 2 - torch.round(raw_last[..., 6])).long()
    inside = mask & (row >= 0) & (row < rows) & (col >= 0) & (col < cols)
    inside[:, 0] = False
    b, v = inside.nonzero(as_tuple=True)
    grid = h.new_zeros(B, rows * cols, C)
    grid[b, row[b, v] * cols + col[b, v]] = h[b, v]                 # two vehicles in one cell: the later one wins
    return grid.view(B, rows, cols, C).permute(0, 3, 1, 2)

def decode_modes(decoder, head, ctx, weights, T_f=TF_STEPS):
    """ctx (B, M, C): one context per mode, fed at every step of a shared LSTM decoder;
    the M raw outputs are moment-matched with `weights` (B, M) into one Gaussian (S7)."""
    B, M, _ = ctx.shape
    h, _ = decoder(ctx.reshape(B * M, 1, -1).expand(-1, T_f, -1))
    return moment_match(head(h).reshape(B, M, T_f, 5), weights)

LANE_HALF = 1.85   # m, half of a 3.7 m lane (assumption)


def manoeuvre_labels(gt, speed0, horizon=TF_STEPS * DT):
    """Labels from the true future: lateral 0 keep / 1 left / 2 right from the 5 s lateral offset
    (+y = right); longitudinal 0 normal / 1 brake if the mean future speed < 0.8 x current speed."""
    y = gt[:, -1, 1]
    lat = torch.where(y < -LANE_HALF, 1, torch.where(y > LANE_HALF, 2, 0))
    lon = (gt[:, -1, 0] / horizon < 0.8 * speed0).long()
    return lat, lon


def manoeuvre_weights(lat_logits, lon_logits):
    """p(lat) x p(lon) -> (B, n_lat * n_lon), mode index = lat * n_lon + lon."""
    return (F.softmax(lat_logits, -1)[:, :, None] * F.softmax(lon_logits, -1)[:, None]).flatten(1)

class PiPNoPlan(nn.Module):
    def __init__(self, norm_stats, d=64, dec=128):
        super().__init__()
        self.register_buffer("mean", torch.tensor(norm_stats["mean"]))
        self.register_buffer("std", torch.tensor(norm_stats["std"]))
        self.encoder = VehicleEncoder(STATE_DIM, 32, d)
        self.branch3 = nn.Conv2d(d, 64, 3, padding=1)
        self.branch5 = nn.Conv2d(d, 64, (5, 3), padding=(2, 1))
        self.merge = nn.Conv2d(128, 64, 3, padding=1)
        ctx = d + 128
        self.op_lat, self.op_lon = nn.Linear(ctx, 3), nn.Linear(ctx, 2)
        self.mode_embed = nn.Embedding(6, 16)
        self.decoder = nn.LSTM(ctx + 16, dec, batch_first=True)
        self.head = nn.Linear(dec, 5)

    def encode(self, states, mask):
        _, h = self.encoder(states)
        grid = social_grid(states[:, -1] * self.std + self.mean, mask, h, rows=25, cols=5)   # M2
        x = F.leaky_relu(torch.cat([self.branch3(grid), self.branch5(grid)], 1), 0.1)
        x = F.leaky_relu(self.merge(x), 0.1)
        ctx = torch.cat([h[:, 0], x.amax((2, 3)), x.mean((2, 3))], -1)
        return ctx, self.op_lat(ctx), self.op_lon(ctx)

    def forward(self, states, mask, dims=None):
        B, _, N, _ = states.shape
        ctx, lat, lon = self.encode(states, mask)
        mode_ctx = torch.cat([ctx[:, None].expand(-1, 6, -1), self.mode_embed.weight[None].expand(B, -1, -1)], -1)
        return decode_modes(self.decoder, self.head, mode_ctx, manoeuvre_weights(lat, lon)), states.new_zeros(B, N - 1, 4), None

    def aux_loss(self, states, mask, dims, gt):
        _, lat, lon = self.encode(states, mask)
        lat_y, lon_y = manoeuvre_labels(gt, (states[:, -1, 0, 2] * self.std[2] + self.mean[2]) * 30.0)
        return F.cross_entropy(lat, lat_y) + F.cross_entropy(lon, lon_y)


def build_model():
    return PiPNoPlan(norm_stats)

model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {n_params:,} parameters")

## 5. Train

**Decisions in this cell:** `fit` with the config budget (S2); best checkpoint by val loss (S4). Loss curves are only comparable within this notebook (S3).

In [ ]:
t0 = time.time()
training_log = fit(model, train_s, val_s, centers, save_dir=str(CKPT_DIR), **HPARAMS)
train_minutes = (time.time() - t0) / 60

log_df = pd.DataFrame(training_log)
display(log_df)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
log_df.plot(x="epoch", y=["train_loss", "val_loss"], marker=".", ax=axes[0], title="Loss")
log_df.plot(x="epoch", y="val_rmse", marker=".", ax=axes[1], title="Val RMSE (m)")
plt.tight_layout(); plt.show()

## 6. Evaluate (best checkpoint) — RMSE 1s…5s overall, per site, zero-neighbour scenes

**Decisions in this cell:** RMSE of the predicted mean at 1-5 s on the test split, overall / per site / target-only scenes (S7).

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / "best_model.pt", map_location=DEVICE, weights_only=True))
model.eval()
test_s = normalize_samples(test_raw, norm_stats)

test_metrics = {}
for label, samples in test_groups(test_s).items():
    dl = create_dataloader(samples, batch_size=32, shuffle=False)
    test_metrics[label] = {**evaluate_trajectory(model, dl, DEVICE), "n": len(samples)}
display(pd.DataFrame(test_metrics).T)

## 7. Collision prediction (optional)
Model's predicted target vs constant-velocity neighbours, every risk metric at its default threshold.

**Decisions in this cell:** neighbours are rolled forward at constant velocity and candidates are the nearest vehicles at the last history step, as in `--mode collisions`; only the target trajectory comes from this model.

In [ ]:
collisions = None
if RUN_COLLISIONS:
    scores, gt, _ = collision_scores(model, loader, test_raw, norm_stats, COLLISION_THRESHOLD)
    collisions = {"threshold": COLLISION_THRESHOLD, "n_pairs": len(gt),
                  "n_gt_collisions": int(gt.sum()), "rows": metric_rows(scores, gt)}
    print(f"{len(gt)} pairs, {int(gt.sum())} GT collisions")
    display(pd.DataFrame(collisions["rows"]).set_index("metric"))

## 8. Save result

**Decisions in this cell:** one JSON per `EXP_NAME`; a rerun overwrites it.

In [ ]:
save_result({
    "name": EXP_NAME,
    "description": DESCRIPTION,
    "model_class": type(model).__name__,
    "n_params": n_params,
    "hparams": HPARAMS,
    "data": {"locations": LOCATIONS, "debug": DEBUG},
    "train_minutes": train_minutes,
    "training_log": training_log,
    "test": test_metrics,
    "collisions": collisions,
})

## 9. Comparison & benchmarking — all results in `experiments/results/`

Ranked by overall test `rmse_avg` (best per column highlighted). Check the `data` column:
debug and full-data runs are listed together but are not comparable.
Also written to `experiments/comparison_rmse.csv` / `comparison_collisions.csv`.

**Decisions in this cell:** compares every saved result; only rows with the same `data` value are like-for-like.

In [ ]:
rmse_df, collision_df = compare()